# Lesson 4: Neural networks — the brain behind deep learning 🧠
**Learning goals**
1. Understand a **neuron**: inputs × weights + bias → activation
2. See how a network **learns** by gradient descent (rolling down a hill)
3. Train a neural network yourself — first on a puzzle a single neuron can't solve, then on handwritten digits

**⏱️ Timing:** one neuron 15' | activation functions 5' | gradient descent 15' | XOR puzzle 10' | digit network 15'

> 💡 Runs on CPU. A GPU is not needed.

## 🏖️ An analogy: should I go to the beach?
You decide based on a few things, and some matter more than others:

| Input | Value (1 = yes, 0 = no) | How much I care (**weight**) |
|---|---|---|
| Is it sunny? | 1 | +4 |
| Are my friends going? | 0 | +3 |
| Do I have homework due? | 1 | −5 |

Add up **value × weight**, plus your general mood (**bias**). If the total is above 0 → go!

That's exactly what an artificial **neuron** does. A neural network is thousands or billions of these connected in **layers**.

In [ ]:
def neuron(inputs, weights, bias):
    total = sum(x * w for x, w in zip(inputs, weights)) + bias
    return total, ("GO to the beach 🏖️" if total > 0 else "stay home 🏠")

sunny, friends, homework = 1, 0, 1
weights = [4, 3, -5]
bias = 0.5          # 👉 a positive bias = "I generally like going out"

print(neuron([sunny, friends, homework], weights, bias))

### 🧩 Exercise 1
1. Try all combinations: sunny but homework due; friends going but not sunny…
2. Change the weights so that you **always** go if your friends are going.
3. **Key question:** you just chose the weights by hand. In machine learning, who chooses them? (👉 the computer, by learning from data!)

## Part 2: Activation functions — the neuron's "decision style"
Instead of a hard yes/no, real neurons pass the total through a smooth **activation function**:
- **Sigmoid:** squashes any number into 0–1 (like a probability)
- **ReLU:** "if negative, output 0; otherwise pass it through" — the most popular today
- **Tanh:** squashes into −1 to 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(-6, 6, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, (name, y) in zip(axes, [("sigmoid", 1 / (1 + np.exp(-x))), ("ReLU", np.maximum(0, x)), ("tanh", np.tanh(x))]):
    ax.plot(x, y, lw=3); ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5); ax.set_title(name)
plt.show()

## Part 3: How does a network learn? Gradient descent ⛰️
### 🌫️ Analogy: lost on a foggy mountain
You want to reach the bottom of a valley, but it's so foggy you can only feel the slope under your feet. Strategy: **take a small step downhill, again and again.**

- The **height** = the **loss** (how wrong the network is)
- Your **position** = the **weights**
- **Step size** = the **learning rate**

Let's find the bottom of a simple valley: loss = (w − 3)². (The answer is w = 3, but the computer doesn't know that.)

In [ ]:
def loss(w):  return (w - 3) ** 2
def slope(w): return 2 * (w - 3)          # the "gradient": which way is uphill, and how steep

def descend(start=-4, learning_rate=0.1, steps=25):
    w, path = start, [start]
    for _ in range(steps):
        w = w - learning_rate * slope(w)     # step DOWNHILL
        path.append(w)
    return path

path = descend()
ws = np.linspace(-5, 10, 200)
plt.plot(ws, loss(ws), color="gray")
plt.plot(path, [loss(w) for w in path], "o-", color="red")
plt.xlabel("weight w"); plt.ylabel("loss"); plt.title(f"ended at w = {path[-1]:.3f}"); plt.show()

### 🧩 Exercise 2: Choosing the step size
Try `learning_rate` = **0.01**, **0.1**, **0.9** and **1.1**. Fill in the table:

| Learning rate | What happened? |
|---|---|
| 0.01 | |
| 0.1 | |
| 0.9 | |
| 1.1 | |

Real networks do exactly this, but with **millions of weights at once** — a valley in millions of dimensions!

In [ ]:
path = descend(learning_rate=1.1)    # 👉 change me
plt.plot(ws, loss(ws), color="gray"); plt.plot(path, [loss(w) for w in path], "o-", color="red")
plt.ylim(0, 60); plt.title(f"ended at w = {path[-1]:.2f}"); plt.show()

## Part 4: The XOR puzzle — why we need layers 🧩
**XOR** = "one or the other, but not both". Example: *I'll be happy if exactly one of my two best friends comes to my party (they don't get along!)*

| Friend A | Friend B | Happy? |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

A **single** neuron can only draw **one straight line** to split yes from no — and no single line can separate these. Let's prove it, then fix it with a **hidden layer**.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([[0.], [1.], [1.], [0.]])

def train(model, steps=3000, lr=0.1):
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(steps):
        loss = nn.functional.binary_cross_entropy(model(X), y)
        opt.zero_grad(); loss.backward(); opt.step()     # backward() works out the slopes for every weight
    return model

one_neuron = train(nn.Sequential(nn.Linear(2, 1), nn.Sigmoid()))
with_hidden = train(nn.Sequential(nn.Linear(2, 4), nn.Tanh(), nn.Linear(4, 1), nn.Sigmoid()), lr=0.5)

print("inputs | target | one neuron | with hidden layer")
for xi, yi, a, b in zip(X, y, one_neuron(X), with_hidden(X)):
    print(f"{xi.tolist()} |   {int(yi)}    |   {a.item():.2f}     |   {b.item():.2f}")

In [ ]:
# Draw what each model "sees": blue = 0, red = 1
grid = torch.tensor([[a, b] for a in np.linspace(-0.5, 1.5, 100) for b in np.linspace(-0.5, 1.5, 100)], dtype=torch.float32)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, m, title in [(axes[0], one_neuron, "one neuron: one straight line"), (axes[1], with_hidden, "hidden layer: curved boundary")]:
    with torch.no_grad():
        ax.scatter(grid[:, 0], grid[:, 1], c=m(grid).squeeze(), cmap="coolwarm", s=4)
    ax.scatter(X[:, 0], X[:, 1], c=y.squeeze(), cmap="coolwarm", edgecolors="black", s=200)
    ax.set_title(title)
plt.show()

**Deep learning** = networks with **many hidden layers**. Each layer builds more complex patterns from the layer before.

## Part 5: A network that reads handwriting ✍️
Remember the 8×8 digits from Lesson 3? Let's solve them with a neural network:
**64 pixels → 32 hidden neurons → 10 outputs (one per digit 0–9)**

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

digits = load_digits()
X_tr, X_te, y_tr, y_te = train_test_split(digits.data / 16, digits.target, test_size=0.3, random_state=0)
X_tr, X_te = torch.tensor(X_tr, dtype=torch.float32), torch.tensor(X_te, dtype=torch.float32)
y_tr, y_te = torch.tensor(y_tr), torch.tensor(y_te)

net = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
print("Number of weights to learn:", sum(p.numel() for p in net.parameters()))

opt = torch.optim.Adam(net.parameters(), lr=0.01)
losses = []
for epoch in range(100):
    loss = nn.functional.cross_entropy(net(X_tr), y_tr)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())

accuracy = (net(X_te).argmax(1) == y_te).float().mean()
plt.plot(losses); plt.xlabel("training step"); plt.ylabel("loss (how wrong)"); plt.title(f"test accuracy: {accuracy:.1%}"); plt.show()

In [ ]:
# Ask the network about one picture — how confident is it for each digit?
i = 7   # 👉 try other pictures
probs = torch.softmax(net(X_te[i:i+1]), dim=1)[0].detach()
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].imshow(X_te[i].reshape(8, 8), cmap="gray_r"); axes[0].set_title(f"true label: {y_te[i].item()}"); axes[0].axis("off")
axes[1].bar(range(10), probs); axes[1].set_xticks(range(10)); axes[1].set_title("network's confidence")
plt.show()

### 🧩 Exercise 3: Network architect
Change the network and record the test accuracy:

| Network | Accuracy |
|---|---|
| `nn.Linear(64, 10)` (no hidden layer) | |
| 32 hidden neurons (as above) | |
| 4 hidden neurons | |
| two hidden layers: `Linear(64,64), ReLU, Linear(64,32), ReLU, Linear(32,10)` | |

### 🚀 Challenge
- Remove the `nn.ReLU()`. Why does the accuracy drop? (Hint: without activations, stacking layers is no better than one layer — all straight lines.)
- ChatGPT-class models have **hundreds of billions** of weights. How many times bigger than our network is that?

## 📌 Summary
- **Neuron** = inputs × weights + bias → activation function
- **Learning** = gradient descent: nudge every weight a little bit downhill on the loss, again and again
- **Hidden layers** let networks learn curved, complex patterns; many layers = **deep learning**
- Every model in the rest of this course — RNNs, Transformers, autoencoders, GANs, diffusion — is a neural network trained this way